In [ ]:
import numpy as np

In [ ]:


class BM:
    def __init__(self, A, B, PI):
        #intialised with empirical probability generally
        self.A = np.array(A, dtype = np.float64)
        self.B = np.array(B, dtype = np.float64)
        self.PI= np.array(PI, dtype = np.float64)
        self.N = self.A.shape[0]
        self.M = self.B.shape[1]
        

    def cal_alpha_beta(self, x):
        #return alpha beta matrix x first sequence 
        T = len(x)
        alpha = np.zeros((T, self.N))
        beta = np.zeros((T, self.N))

        #forward 
        #alpha calculation
        alpha[0,:] = self.PI*self.B[:, x[0]];
        for t in range(1, T):
            alpha[t,:] = (alpha[t-1,:]@self.A)*self.B[:,x[t]]
            
        #beta calculation j->i->x[t+1]
        beta[T-1,:] =1
        for t in range(T-2, -1, -1):
            beta[t,:] = np.sum((self.B[:,x[t+1]]*beta[t+1, :])*self.A, axis = 1)
        
        return alpha, beta
        
    def cal_gamma_xi(self, alpha, beta, x):
        #return gamma, xi for the given sequence x
        T = len(x);
        gamma = np.zeros((T, self.N)) #t, xt
        xi = np.zeros((T-1, self.N, self.N)) #t, xt, xt+1
        
        #gamma calculation
        for t in range(T):
            denom = np.sum(alpha[t,:]*beta[t,:])
            gamma[t,:] = (alpha[t,:]*beta[t,:])/denom
        #xi calculation
        for t in range(T-1):
            denom = np.sum(
                (alpha[t,:][:,None]*self.A)*(self.B[:, x[t+1]]*beta[t+1, :])
            )
            xi[t] = (alpha[t,:][:,None]*self.A)*(self.B[:, x[t+1]]*beta[t+1, :])/denom

        return gamma, xi
        
    def fit(self, X, n_iter = 20):

        #global counts for the calculation of A, B, PI
    
        # X - contain multiple sentence sequences tokenized with serial no
        #this will used to calculate BM latent parameter
        for iter in range(n_iter):
            A_cnt = np.zeros((self.N, self.N))
            B_cnt = np.zeros((self.N, self.M))
            PI_cnt = np.zeros(self.N)
            denom_A = np.zeros(self.N)
            denom_B = np.zeros(self.N)
            total_llhood = 0
            
            
            for x in X:
                alpha, beta = self.cal_alpha_beta(x)
                gamma, xi = self.cal_gamma_xi(alpha, beta, x)
                T = len(x)
                #accumiulate to global count over all X
                #using gamma, xi

                #PI
                PI_cnt += gamma[0]
                #A_cnt
                A_cnt += np.sum(xi, axis=0)
                denom_A += np.sum(gamma[:-1], axis=0)
                #B_cnt
                for t in range(T):
                    B_cnt[:, x[t]] += gamma[t,:]
                    denom_B += gamma[t,:]

                #seq_prob
                seq_prob = np.sum(alpha[-1,:])
                total_llhood+=np.log(seq_prob + 1e-300)
            #update A, B, PI
            self.PI = PI_cnt / len(X)
            self.A = A_cnt / denom_A[:, None]
            self.B = B_cnt / denom_B[:, None]

            print(
                f"Iteration {iter+1} "
                f"Log-Likelihood = {total_llhood:.4f}"
            )

            #next iteration
            
        return self
        
        
        
        